<a href="https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring, and I frame it primarily as a ranking/scoring task type.

The main decision is not simply whether a page is declining. The decision is which pages a content or SEO team should review first when their time and resources are limited. Therefore, the output should be a priority score that ranks pages from highest to lowest review priority. A classification model may later be used as one input into this score, but the final business task is ranking pages for action. This matches the internship guidance, where the expected output is a ranked review queue with scores, actions, reason codes, and confidence labels.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# My ML task framing
ml_task = "Ranking / Scoring"

print("Chosen ML task:", ml_task)
print("Decision: Which pages should be reviewed first?")
print("Output: Ranked priority queue of pages")

Chosen ML task: Ranking / Scoring
Decision: Which pages should be reviewed first?
Output: Ranked priority queue of pages


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter dataset, I will use trend_direction == "down" as a proxy label for decline risk. This means that a page is labelled positive when its observed trend direction in the current dataset is "down". However, this is not an ideal future-looking target because the label comes from the current observation window rather than measuring what happens after a decision point. Therefore, I will treat it as a beginner proxy for testing whether observable page signals can help identify pages showing decline-related risk. The stronger long-term version of this project would use a future observed outcome, such as using signals from the previous 90 days to predict whether a page experiences sustained decline or recovery during the following 30 days.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Create the starter proxy label
df["is_declining_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

# Check the class distribution
print(df["is_declining_proxy"].value_counts())

print("\nPercentage declining:")
print(
    round(
        df["is_declining_proxy"].mean() * 100,
        2
    ),
    "%"
)

is_declining_proxy
1    16262
0    13738
Name: count, dtype: int64

Percentage declining:
54.21 %


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric will be Precision@K, specifically Precision@50 for the starter dataset. The reason is that the real decision involves limited review capacity: a content team cannot investigate every page, so the most important question is how many useful candidates appear near the top of the ranked list. Precision@50 measures the proportion of the top 50 recommended pages that match the chosen positive proxy label.

For example, a Precision@50 score of 0.70 would mean that approximately 35 of the top 50 recommended pages match the selected positive label. This metric is more aligned with the real decision than accuracy because the project is focused on prioritizing the highest-value pages for review rather than treating every page equally.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(y_true, y_scores, k=50):
    top_k_indices = y_scores.argsort()[::-1][:k]
    return y_true.iloc[top_k_indices].mean()

print("Primary evaluation metric: Precision@50")
print(
    "Meaning: Of the top 50 recommended pages, "
    "what proportion are positive according to the chosen proxy?"
)


Primary evaluation metric: Precision@50
Meaning: Of the top 50 recommended pages, what proportion are positive according to the chosen proxy?


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content page. Each row in the starter dataset represents a pseudonymized content item and contains observable signals describing its search performance, traffic, content characteristics, freshness, position, engagement, and trend information. The ranking system will assign each page a priority score so that pages can be ordered according to which ones should be reviewed first.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nOne row represents: one content page")

print("\nDuplicate content IDs:")
print(df["content_id"].duplicated().sum())

print("\nNumber of unique content pages:")
print(df["content_id"].nunique())

Dataset shape: (30000, 44)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



One row represents: one content page

Duplicate content IDs:
0

Number of unique content pages:
30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can identify some obvious candidates, such as pages with high impressions and declining performance. However, page review priority depends on multiple signals that may interact in different ways. For example, impressions, clicks, CTR, average position, content age, freshness, engagement, sessions, content depth, and trend direction may all contribute to whether a page deserves attention.

A single if-statement would require manually deciding how much each signal matters and how they should interact. This becomes difficult when a page has mixed signals, such as high visibility but weak CTR, strong position but declining sessions, or old content with low engagement. A ranking or ML approach can combine multiple observable signals and learn more complex patterns than a single fixed rule.

However, ML must earn its place by outperforming a transparent baseline. I will therefore compare any ML-based approach against a simple rule-based scoring baseline rather than assuming that a more complex model is automatically better.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the availability of important signals

important_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction"
]

print("Important features available:")
for feature in important_features:
    if feature in df.columns:
        print(f"✓ {feature}")
    else:
        print(f"✗ {feature} not found")

print(
    "\nConclusion: Page priority depends on multiple signals, "
    "not a single condition."
)


Important features available:
✓ impressions_90d
✓ clicks_90d
✓ sessions_90d
✓ content_age_days
✓ trend_direction

Conclusion: Page priority depends on multiple signals, not a single condition.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.